# Colab Inference Endpoint (Gemma + LoRA)

This notebook starts a FastAPI server in Colab and exposes it with ngrok.

It uses model files downloaded from your Google Drive folder link.

Expected files inside that folder:
- `gemma-keras-gemma_1.1_instruct_2b_en-v4/`
- `gemma_lora.weights.h5`

In [1]:
!pip -q install tensorflow keras keras-nlp fastapi "uvicorn[standard]" pyngrok requests gdown

## 1) Download model files from Google Drive

Base model files are downloaded from your shared folder, and fine-tuned LoRA weights are downloaded from your direct file link.

In [15]:
# Set your ngrok auth token for a stable tunnel
# NGROK_AUTH_TOKEN = ""  # e.g. "2abc..."  <--- Cleared for now due to invalid format


In [8]:
from pathlib import Path
import os



# Optional: mount Google Drive if you also keep files in your own Drive
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# Shared Google Drive folder for full base model files
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/150sKqJC2BV4_VLm04pH-IlBMl8xvmiEC?usp=drive_link"
DOWNLOAD_DIR = Path("/content/model_bundle")

# Direct Google Drive file link for fine-tuned LoRA weights
FINE_TUNE_WEIGHTS_URL = "https://drive.google.com/file/d/141m-UgnkzCXydq-50-0EHGxOs5bSfBi-/view?usp=sharing"
LORA_DEST = Path("/content/gemma_lora.weights.h5")

import gdown
import h5py
import keras_nlp
import tensorflow as tf
from keras import mixed_precision

# Low-memory mode for Colab GPUs (T4/L4/A100 all support this path).
mixed_precision.set_global_policy("mixed_float16")

# Download base model bundle from Drive folder (cached after first run)
if not DOWNLOAD_DIR.exists() or not any(DOWNLOAD_DIR.rglob("*")):
    print("Downloading model bundle from Google Drive folder...")
    gdown.download_folder(url=DRIVE_FOLDER_URL, output=str(DOWNLOAD_DIR), quiet=False)
else:
    print(f"Using cached files in: {DOWNLOAD_DIR}")

# Download fine-tuned weights from direct file link
if not LORA_DEST.exists():
    print("Downloading fine-tuned LoRA weights from Google Drive file link...")
    gdown.download(url=FINE_TUNE_WEIGHTS_URL, output=str(LORA_DEST), fuzzy=True, quiet=False)
else:
    print(f"Using cached LoRA weights: {LORA_DEST}")

# Candidate roots to search for the Gemma preset
search_roots = [DOWNLOAD_DIR, Path("/content")]
if MOUNT_DRIVE:
    search_roots.append(Path("/content/drive/MyDrive"))

def is_preset_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    has_config = (p / "config.json").exists()
    has_weights = (p / "model.weights.h5").exists()
    has_tokenizer_json = (p / "tokenizer.json").exists()
    has_spm = (p / "assets/tokenizer/vocabulary.spm").exists()
    return has_config and has_weights and (has_tokenizer_json or has_spm)

def has_base_embedding_weights(weights_path: Path) -> bool:
    if not weights_path.exists() or weights_path.stat().st_size < 100 * 1024 * 1024:
        return False
    try:
        found_token_embedding = False
        found_embeddings_var = False
        with h5py.File(weights_path, "r") as f:
            def visitor(name, obj):
                nonlocal found_token_embedding, found_embeddings_var
                lname = name.lower()
                if "token_embedding" in lname:
                    found_token_embedding = True
                if "embeddings" in lname:
                    found_embeddings_var = True
            f.visititems(visitor)
        return found_token_embedding and found_embeddings_var
    except Exception:
        return False

preset_candidates = []
for root in search_roots:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if is_preset_dir(p):
            preset_candidates.append(p)

seen = set()
preset_candidates = [p for p in preset_candidates if not (str(p) in seen or seen.add(str(p)))]

assert LORA_DEST.exists(), (
    "Could not download LoRA weights from Google Drive file link. "
    f"Checked path: {LORA_DEST}"
)

if not preset_candidates:
    raise AssertionError(
        "Could not find Gemma preset folder. "
        "Make sure your Drive folder contains the extracted preset directory "
        "(with config.json + model.weights.h5 + tokenizer files)."
    )

candidate_info = []
for p in preset_candidates:
    wp = p / "model.weights.h5"
    size_gb = round(wp.stat().st_size / (1024**3), 2) if wp.exists() else 0.0
    strict_asset = (p / "assets/tokenizer/vocabulary.spm").exists()
    embedding_ok = has_base_embedding_weights(wp)
    name_has_gemma = "gemma" in p.name.lower()
    candidate_info.append((p, strict_asset, embedding_ok, name_has_gemma, size_gb))

def candidate_sort_key(item):
    p, strict_asset, embedding_ok, name_has_gemma, size_gb = item
    return (0 if strict_asset else 1, 0 if embedding_ok else 1, 0 if name_has_gemma else 1, -size_gb, len(str(p)))

chosen_preset = sorted(candidate_info, key=candidate_sort_key)[0][0]

PRESET_PATH = str(chosen_preset)
LORA_WEIGHTS_PATH = str(LORA_DEST)

assert os.path.isdir(PRESET_PATH), f"Invalid PRESET_PATH: {PRESET_PATH}"
assert os.path.isfile(LORA_WEIGHTS_PATH), f"Invalid LORA_WEIGHTS_PATH: {LORA_WEIGHTS_PATH}"

print("Selected preset:", PRESET_PATH)
print("Selected base weights size (GB):", round((Path(PRESET_PATH) / "model.weights.h5").stat().st_size / (1024**3), 2))
print("LoRA weights:", LORA_WEIGHTS_PATH)

# Load model with reduced precision to lower GPU memory use.
print("Loading base preset (float16)...")
model = keras_nlp.models.GemmaCausalLM.from_preset(PRESET_PATH, dtype="float16")
print("Enabling LoRA and loading fine-tuned weights...")
model.backbone.enable_lora(rank=16)
model.backbone.load_lora_weights(LORA_WEIGHTS_PATH)
print("Model is ready.")

Using cached files in: /content/model_bundle
Using cached LoRA weights: /content/gemma_lora.weights.h5
Selected preset: /content/model_bundle
Selected base weights size (GB): 4.67
LoRA weights: /content/gemma_lora.weights.h5
Loading base preset (float16)...
Enabling LoRA and loading fine-tuned weights...


AttributeError: 'GemmaCausalLM' object has no attribute 'load_lora_weights'

In [ ]:
import gc
import tensorflow as tf

# Delete big objects if they exist
for v in ["model"]:
    if v in globals():
        del globals()[v]

gc.collect()
tf.keras.backend.clear_session()
gc.collect()

print("Cleared session and ran garbage collection.")

## 2) Model check (already loaded in Cell 4)

In [13]:
# Model is loaded in Cell 4.
# This check helps confirm the model object is available before starting the API server.
assert "model" in globals(), "Model is not loaded. Run Cell 4 first."
print("Model exists in memory. Ready for server startup.")

Model exists in memory. Ready for server startup.


## 3) Start FastAPI server and expose public URL

In [16]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import threading
import time
from pyngrok import ngrok
from pyngrok.exception import PyngrokNgrokError
import os

PORT = 8000

# Global variables for managing the server thread
_uvicorn_server = None
_server_thread = None
_stop_event = threading.Event()

app = FastAPI(title="Gemma LoRA Inference API")

# Safer defaults for low-memory GPUs
MAX_ALLOWED_LENGTH = 96
DEFAULT_MAX_LENGTH = 48

class GenerateRequest(BaseModel):
    prompt: str
    max_length: int = DEFAULT_MAX_LENGTH

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/generate")
def generate(req: GenerateRequest):
    max_len = min(max(16, req.max_length), MAX_ALLOWED_LENGTH)
    output = model.generate(
        req.prompt,
        max_length=max_len,
    )
    text = output if isinstance(output, str) else str(output)
    return {"text": text, "max_length_used": max_len}

local_url = f"http://localhost:{PORT}"
print(f"Local endpoint: {local_url}")
print(f"Health URL: {local_url}/health")
print(f"Generate URL: {local_url}/generate")

public_url = None

def run_server_target():
    global _uvicorn_server
    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
    _uvicorn_server = uvicorn.Server(config)
    _uvicorn_server.run()

def start_or_restart_server():
    global _uvicorn_server, _server_thread, _stop_event, public_url

    # Attempt to stop existing server if it's running
    if _uvicorn_server and _server_thread and _server_thread.is_alive():
        print("Stopping existing Uvicorn server...")
        _uvicorn_server.should_exit = True  # Signal the server to stop
        _server_thread.join(timeout=10) # Give more time for the thread to stop

        if _server_thread.is_alive():
            print("Warning: Existing server thread did not stop gracefully. It might still occupy the port.")

        # Add a short delay after trying to stop the server
        time.sleep(2) # Give the OS a moment to release the port

    # Disconnect ngrok if it was connected
    if public_url:
        print("Disconnecting ngrok tunnel...")
        try:
            ngrok.disconnect(public_url)
        except Exception as e:
            print(f"Error disconnecting ngrok: {e}")
        public_url = None

    # Reset for a fresh start
    _uvicorn_server = None
    _server_thread = None
    _stop_event.clear()
    public_url = None

    print("Starting new Uvicorn server...")
    _server_thread = threading.Thread(target=run_server_target, daemon=True)
    _server_thread.start()

    # (Re-)connect ngrok if auth token is available
    if NGROK_AUTH_TOKEN:
        try:
            ngrok.set_auth_token(NGROK_AUTH_TOKEN)
            public_url = ngrok.connect(PORT).public_url
            print(f"Public ngrok URL: {public_url}")
            print(f"Health URL: {public_url}/health")
            print(f"Generate URL: {public_url}/generate")
        except PyngrokNgrokError as e:
            print(f"Error connecting ngrok: {e}")
            print("Please check your NGROK_AUTH_TOKEN and try again.")
    else:
        print("NGROK_AUTH_TOKEN is not set. Public URL not available.")

# Start or restart the server when this cell is executed
start_or_restart_server()

Local endpoint: http://localhost:8000
Health URL: http://localhost:8000/health
Generate URL: http://localhost:8000/generate
Starting new Uvicorn server...


INFO:     Started server process [8937]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


Public ngrok URL: https://vehicular-grueling-yodel.ngrok-free.dev
Health URL: https://vehicular-grueling-yodel.ngrok-free.dev/health
Generate URL: https://vehicular-grueling-yodel.ngrok-free.dev/generate


## 4) Quick endpoint test

In [17]:
import requests

# Use local_url for local testing, or public_url if available for external testing
# Ensure public_url is defined from the server cell if ngrok is active.

# Fallback to local_url if public_url is not set (e.g., ngrok token not provided)
# This assumes the FastAPI server cell has already run and populated public_url
if 'public_url' in globals() and public_url:
    base_url = public_url
    print(f"Testing public ngrok URL: {base_url}")
else:
    base_url = "http://localhost:8000"
    print(f"Testing local endpoint: {base_url}")

health = requests.get(f"{base_url}/health", timeout=30)
print("Health:", health.status_code, health.text)

payload = {
    "prompt": "Say hello in one short sentence.",
    "max_length": 64,
}
resp = requests.post(f"{base_url}/generate", json=payload, timeout=120)
print("Generate status:", resp.status_code)
print(resp.json())

Testing public ngrok URL: https://vehicular-grueling-yodel.ngrok-free.dev
INFO:     34.106.113.47:0 - "GET /health HTTP/1.1" 200 OK
Health: 200 {"status":"ok"}
INFO:     34.106.113.47:0 - "POST /generate HTTP/1.1" 200 OK
Generate status: 200
{'text': 'Say hello in one short sentence.\n\nHello! 👋', 'max_length_used': 64}


## Notes

- Keep the notebook session running while you use the endpoint.
- If Colab disconnects, rerun the server cell to get a new URL.
- Do not expose this endpoint publicly without adding auth/rate limiting for production use.